# Tutorial 04: DAG Workflow End-to-End

This notebook runs a real DAG (Directed Acyclic Graph) workflow end-to-end using a live LLM.
We will load, validate, and execute AWP workflows that use the DAG engine — the topological
execution engine for A0-A1 autonomy levels.

**Prerequisites:**
- `pip install -e reference/python/`
- An LLM API key (OpenRouter, Ollama, or any OpenAI-compatible endpoint)

## 1. Provider Setup

Choose your LLM provider below. Supported options:
- **Ollama** — local, no API key needed (install from ollama.com)
- **OpenRouter** — cloud, requires `OPENROUTER_API_KEY`
- **Custom API** — any OpenAI-compatible endpoint, requires `LLM_API_KEY` + `LLM_BASE_URL`

In [1]:
# ============================================================
# Provider Selection — choose ONE of: "ollama", "openrouter", "custom"
# ============================================================
PROVIDER = "openrouter"  # <-- change this

# --- Ollama (local) -------------------------------------------
OLLAMA_MODEL = "qwen3:1.7b"            # any model you've pulled
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# --- OpenRouter (cloud) ---------------------------------------
OPENROUTER_API_KEY = ""                 # paste your key or set env var OPENROUTER_API_KEY
OPENROUTER_MODEL = "openrouter/google/gemini-2.5-flash"

# --- Custom OpenAI-compatible API -----------------------------
CUSTOM_API_KEY = ""                     # paste your key or set env var LLM_API_KEY
CUSTOM_BASE_URL = ""                    # e.g. "https://api.openai.com/v1"
CUSTOM_MODEL = ""                       # e.g. "gpt-4o"

# ==============================================================
# DO NOT EDIT BELOW — wires up the selected provider
# ==============================================================
import os

if PROVIDER == "ollama":
    os.environ["LLM_API_KEY"] = "ollama"
    os.environ["LLM_BASE_URL"] = OLLAMA_BASE_URL
    MODEL = f"ollama/{OLLAMA_MODEL}"
    print(f"Using Ollama  model={OLLAMA_MODEL}  url={OLLAMA_BASE_URL}")

elif PROVIDER == "openrouter":
    key = OPENROUTER_API_KEY or os.getenv("OPENROUTER_API_KEY", "")
    if not key:
        raise ValueError("Set OPENROUTER_API_KEY above or as an environment variable")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = "https://openrouter.ai/api/v1"
    MODEL = OPENROUTER_MODEL
    print(f"Using OpenRouter  model={OPENROUTER_MODEL}")

elif PROVIDER == "custom":
    key = CUSTOM_API_KEY or os.getenv("LLM_API_KEY", "")
    url = CUSTOM_BASE_URL or os.getenv("LLM_BASE_URL", "")
    if not key:
        raise ValueError("Set CUSTOM_API_KEY above or LLM_API_KEY as an environment variable")
    if not url:
        raise ValueError("Set CUSTOM_BASE_URL above or LLM_BASE_URL as an environment variable")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = url
    MODEL = CUSTOM_MODEL
    print(f"Using custom API  model={CUSTOM_MODEL}  url={url}")

else:
    raise ValueError(f"Unknown PROVIDER '{PROVIDER}'. Use 'ollama', 'openrouter', or 'custom'.")

# Set the model env var so WorkflowRunner picks it up
os.environ["LLM_MODEL"] = MODEL
print(f"LLM_MODEL={MODEL}")
print(f"LLM_BASE_URL={os.environ['LLM_BASE_URL']}")

Using OpenRouter  model=openrouter/google/gemini-2.5-flash
LLM_MODEL=openrouter/google/gemini-2.5-flash
LLM_BASE_URL=https://openrouter.ai/api/v1


## 2. What is a DAG Workflow?

A **DAG (Directed Acyclic Graph)** workflow defines agents as nodes and their dependencies as edges.
The AWP DAG engine executes agents in **topological order** — each agent runs only after all its
dependencies have completed.

Key properties:
- **Deterministic execution order** — the graph structure determines which agents run first
- **State sharing** — agents can share outputs with downstream agents via `share_output`
- **Sequential or parallel** — independent agents can run in parallel; dependent agents run sequentially
- **A0-A1 autonomy** — the workflow structure is fully defined at design time

### The Research Pipeline Example

The `02-research-pipeline` workflow has three agents in a linear chain:

```
planner --> researcher --> writer
```

1. **planner** — generates research questions and a search strategy
2. **researcher** — investigates the questions (depends on planner output)
3. **writer** — produces a final report (depends on researcher findings)

Each agent shares specific output fields with the next agent in the pipeline.

In [2]:
# Visualize the DAG structure
print("Research Pipeline DAG:")
print()
print("  [planner]")
print("      |")
print("      | shares: research_questions, search_strategy")
print("      v")
print("  [researcher]")
print("      |")
print("      | shares: findings, sources")
print("      v")
print("  [writer]")
print("      |")
print("      | shares: report")
print("      v")
print("  (done)")

Research Pipeline DAG:

  [planner]
      |
      | shares: research_questions, search_strategy
      v
  [researcher]
      |
      | shares: findings, sources
      v
  [writer]
      |
      | shares: report
      v
  (done)


## 3. Load and Validate the Workflow

Before executing, we parse the YAML manifest, validate the graph structure, and check
the autonomy level compliance.

In [3]:
from pathlib import Path
from awp.parser import parse_manifest, parse_agent
from awp.validator import validate_graph, check_compliance, AutonomyLevel

# Use absolute path to the project root
PROJECT = Path("/home/shumway/projects/agent-workflow-protocol")
workflow_path = PROJECT / "examples/02-research-pipeline/workflow.awp.yaml"

# Parse the manifest
manifest = parse_manifest(workflow_path)
print(f"Workflow: {manifest.workflow.name}")
print(f"Version:  {manifest.workflow.version}")
print(f"Description: {manifest.workflow.description}")
print(f"Tags: {manifest.workflow.tags}")

Workflow: research-pipeline
Version:  1.0.0
Description: Multi-agent research pipeline demonstrating A1 Adaptive autonomy level with state sharing
Tags: ['example', 'a1', 'adaptive', 'research']


In [4]:
# Validate the graph structure
graph_result = validate_graph(manifest.orchestration)
print(f"Graph valid: {graph_result.valid}")
if graph_result.errors:
    print(f"Errors: {graph_result.errors}")
if graph_result.warnings:
    print(f"Warnings: {graph_result.warnings}")
print()

# Show all nodes and their dependencies
print("Graph nodes:")
for node in manifest.orchestration.graph:
    deps = node.depends_on if node.depends_on else []
    shared = node.share_output if node.share_output else []
    print(f"  {node.id}")
    print(f"    depends_on:   {deps}")
    print(f"    share_output: {shared}")

Graph valid: True

Graph nodes:
  planner
    depends_on:   []
    share_output: ['research_questions', 'search_strategy']
  researcher
    depends_on:   ['planner']
    share_output: ['findings', 'sources']
  writer
    depends_on:   ['researcher']
    share_output: ['report']


In [5]:
# Load agent definitions and check autonomy compliance
workflow_dir = PROJECT / "examples/02-research-pipeline"
agents = {}
agents_dir = workflow_dir / "agents"
for agent_dir in sorted(agents_dir.iterdir()):
    agent_yaml = agent_dir / "agent.awp.yaml"
    if agent_yaml.exists():
        agent = parse_agent(agent_yaml)
        agent_id = agent.identity.id
        agents[agent_id] = agent
        print(f"Agent '{agent_id}': role={agent.identity.role}")

print()

# Check compliance — what autonomy level does this workflow achieve?
compliance = check_compliance(manifest, agents, workflow_path=workflow_dir)
print(f"Autonomy level achieved: {compliance.level.name}")
print(f"Checks: {compliance.checks}")

Agent 'planner': role=research_planner
Agent 'researcher': role=research_analyst
Agent 'writer': role=report_writer

Autonomy level achieved: A0_PRESCRIBED
Checks: {'manifest_present': True, 'awp_version_set': True, 'workflow_name_valid': True, 'at_least_one_agent': True, 'agent_planner_has_contract': True, 'agent_researcher_has_contract': True, 'agent_writer_has_contract': True}


## 4. Run the DAG Workflow

Now we execute the research pipeline end-to-end. The `WorkflowRunner` takes a directory path
(not a YAML file path) and handles parsing, agent instantiation, and execution.

The runner uses the `LLM_API_KEY`, `LLM_BASE_URL`, and `LLM_MODEL` environment variables
we configured in section 1.

In [6]:
import logging

# Enable INFO logging so we can see the execution flow
logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")

from awp.runtime import WorkflowRunner

runner = WorkflowRunner(PROJECT / "examples/02-research-pipeline")
result = runner.run("Research the latest trends in renewable energy storage technology")

print("\n--- Workflow Complete ---")
print(f"Result keys: {list(result.keys())}")

awp.runtime.runner: Running workflow 'research-pipeline' [run_id=bfaaec5593d0] with 3 agents in 3 levels


awp.runtime.runner: Level 0: [planner]


httpx: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 401 Unauthorized"


awp.runtime.agent: Agent planner LLM error: Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


awp.runtime.runner:   Completed: planner (0.4s)


...

httpx: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 401 Unauthorized"


awp.runtime.agent: Agent writer LLM error: Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


awp.runtime.runner:   Completed: writer (0.1s)


awp.runtime.state_persistence: Final state saved: /home/shumway/projects/agent-workflow-protocol/examples/02-research-pipeline/data/state/final.json



--- Workflow Complete ---
Result keys: ['task', 'planner', 'researcher', 'writer']


In [7]:
# Display the full result dict
import json

print(json.dumps(result, indent=2, default=str))

{
  "task": "Research the latest trends in renewable energy storage technology",
  "planner": {
    "error": "Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401",
    "confidence": 0.0
  },
  "researcher": {
    "error": "Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401",
    "confidence": 0.0
  },
  "writer": {
    "error": "Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401",
    "confidence": 0.0
  }
}


In [8]:
# Access individual agent outputs
for agent_name in ["planner", "researcher", "writer"]:
    if agent_name in result:
        agent_result = result[agent_name]
        confidence = agent_result.get("confidence", "N/A")
        print(f"=== {agent_name} (confidence: {confidence}) ===")
        # Show first 500 chars of each agent's output
        for key, value in agent_result.items():
            if key == "confidence":
                continue
            text = str(value)
            if len(text) > 500:
                text = text[:500] + "..."
            print(f"  {key}: {text}")
        print()
    else:
        print(f"=== {agent_name}: not found in result ===")

=== planner (confidence: 0.0) ===
  error: Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

=== researcher (confidence: 0.0) ===
  error: Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

=== writer (confidence: 0.0) ===
  error: Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401



In [9]:
# Show state sharing between agents
# The planner's shared fields (research_questions, search_strategy) become
# available to the researcher via the state dict. Similarly, the researcher's
# shared fields (findings, sources) flow to the writer.

print("State sharing chain:")
print()
if "planner" in result:
    planner_out = result["planner"]
    shared_to_researcher = {k: v for k, v in planner_out.items()
                            if k in ("research_questions", "search_strategy")}
    print(f"planner -> researcher:")
    for k, v in shared_to_researcher.items():
        text = str(v)[:300] + ("..." if len(str(v)) > 300 else "")
        print(f"  {k}: {text}")
    print()

if "researcher" in result:
    researcher_out = result["researcher"]
    shared_to_writer = {k: v for k, v in researcher_out.items()
                        if k in ("findings", "sources")}
    print(f"researcher -> writer:")
    for k, v in shared_to_writer.items():
        text = str(v)[:300] + ("..." if len(str(v)) > 300 else "")
        print(f"  {k}: {text}")
    print()

if "writer" in result:
    writer_out = result["writer"]
    if "report" in writer_out:
        print(f"writer final report (first 500 chars):")
        print(str(writer_out["report"])[:500])

State sharing chain:

planner -> researcher:

researcher -> writer:



## 5. Inspect Observability Output

The DAG engine produces logs and traces during execution. Let us check what
observability data was generated.

In [10]:
# Check for log files generated by the workflow
logs_dir = PROJECT / "examples/02-research-pipeline/logs"
if logs_dir.exists():
    log_files = list(logs_dir.iterdir())
    print(f"Log files found: {len(log_files)}")
    for f in sorted(log_files)[-5:]:  # Show last 5 log files
        print(f"  {f.name} ({f.stat().st_size} bytes)")
else:
    print("No logs directory found")

# Check for state persistence data
state_dir = PROJECT / "examples/02-research-pipeline/data/state"
if state_dir.exists():
    state_files = list(state_dir.rglob("*"))
    print(f"\nState files found: {len(state_files)}")
    for f in sorted(state_files)[-5:]:
        if f.is_file():
            print(f"  {f.relative_to(state_dir)} ({f.stat().st_size} bytes)")
else:
    print("\nNo state directory found")

Log files found: 31
  e2e_run_20260328_022656.log (1189 bytes)
  e2e_run_20260328_022720.log (1189 bytes)
  e2e_run_20260328_023305.log (1189 bytes)
  e2e_run_20260328_024326.log (1189 bytes)
  validation_20260324_203843.json (774 bytes)

State files found: 4
  final.json (885 bytes)
  planner.json (420 bytes)
  researcher.json (670 bytes)
  writer.json (909 bytes)


In [11]:
# Show agent execution order from the graph topology
# The DAG engine resolves this automatically via topological sort
from collections import deque

graph = manifest.orchestration.graph
adj = {node.id: (node.depends_on or []) for node in graph}

# Compute topological order (Kahn's algorithm)
in_degree = {nid: 0 for nid in adj}
for nid, deps in adj.items():
    for dep in deps:
        pass  # in_degree tracks how many depend on each node
# Simpler: just count incoming edges
in_degree = {nid: len(deps) for nid, deps in adj.items()}

order = []
queue = deque([nid for nid, deg in in_degree.items() if deg == 0])
while queue:
    node = queue.popleft()
    order.append(node)
    for nid, deps in adj.items():
        if node in deps:
            in_degree[nid] -= 1
            if in_degree[nid] == 0:
                queue.append(nid)

print("Topological execution order:")
for i, agent_id in enumerate(order, 1):
    deps = adj[agent_id]
    dep_str = f" (after: {', '.join(deps)})" if deps else " (root)"
    print(f"  {i}. {agent_id}{dep_str}")

print()
print("Execution config:")
exec_cfg = manifest.orchestration.execution
print(f"  Mode: {exec_cfg.mode}")
print(f"  Timeout per agent: {exec_cfg.timeout.per_agent}s")
print(f"  Timeout total: {exec_cfg.timeout.total}s")
print(f"  Max parallel agents: {exec_cfg.max_parallel_agents}")
print(f"  Error handling: {exec_cfg.error_handling.default}")

Topological execution order:
  1. planner (root)
  2. researcher (after: planner)
  3. writer (after: researcher)

Execution config:
  Mode: sequential
  Timeout per agent: 60s
  Timeout total: 300s
  Max parallel agents: 4
  Error handling: continue


## 6. Run Hello World (Simplest Case)

The simplest possible DAG workflow: a single agent with no dependencies.
This is an A0 (Prescribed) workflow — fully deterministic, one node, no branching.

In [12]:
# Load and inspect the hello-world workflow
hello_manifest = parse_manifest(PROJECT / "examples/01-hello-world/workflow.awp.yaml")
print(f"Workflow: {hello_manifest.workflow.name}")
print(f"Graph nodes: {[n.id for n in hello_manifest.orchestration.graph]}")
print(f"State sharing: {hello_manifest.state.sharing.strategy}")
print()

# Run it
hello_runner = WorkflowRunner(PROJECT / "examples/01-hello-world")
hello_result = hello_runner.run("Say hello to the AWP community")

print("\n--- Hello World Result ---")
print(json.dumps(hello_result, indent=2, default=str))

awp.runtime.runner: Running workflow 'hello-world' [run_id=5289a1c7c0b5] with 1 agents in 1 levels


awp.runtime.runner: Level 0: [greeter]


httpx: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 401 Unauthorized"


awp.runtime.agent: Agent greeter LLM error: Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


awp.runtime.runner:   Completed: greeter (0.1s)


awp.runtime.state_persistence: Final state saved: /home/shumway/projects/agent-workflow-protocol/examples/01-hello-world/data/state/final.json


Workflow: hello-world
Graph nodes: ['greeter']
State sharing: full


--- Hello World Result ---
{
  "task": "Say hello to the AWP community",
  "greeter": {
    "error": "Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401",
    "confidence": 0.0
  }
}


## 7. Understanding State Sharing

State sharing is how agents pass data to downstream agents in a DAG workflow.

- **`share_output`** — declared on each graph node; specifies which keys from the agent's
  output dict are injected into the shared state.
- **Downstream agents** receive these values in their `state` dict when they execute.
- **Sharing strategies**: `full` (all outputs shared), `selective` (only declared fields shared).

### How data flows

```
planner.run(state={"task": "..."})
  -> returns {"planner": {"research_questions": [...], "search_strategy": "...", "confidence": 0.9}}
  -> share_output: ["research_questions", "search_strategy"]
  -> state now contains: {"task": "...", "research_questions": [...], "search_strategy": "..."}

researcher.run(state={"task": "...", "research_questions": [...], "search_strategy": "..."})
  -> returns {"researcher": {"findings": "...", "sources": [...], "confidence": 0.85}}
  -> share_output: ["findings", "sources"]
  -> state now contains: {... + "findings": "...", "sources": [...]}

writer.run(state={...all accumulated state...})
  -> returns {"writer": {"report": "...", "confidence": 0.9}}
```

In [13]:
# Demonstrate state sharing from the actual result
# Each agent's share_output fields accumulate into the shared state dict

print("=== State Sharing Analysis ===")
print()

# Walk the graph and show what each node shares
for node in manifest.orchestration.graph:
    shared_fields = node.share_output or []
    agent_output = result.get(node.id, {})

    print(f"Node: {node.id}")
    print(f"  Declared share_output: {shared_fields}")

    # Show the actual shared values
    for field in shared_fields:
        value = agent_output.get(field, "<not present>")
        text = str(value)
        if len(text) > 200:
            text = text[:200] + "..."
        print(f"  -> {field} = {text}")

    # Show dependencies (what this node received)
    deps = node.depends_on or []
    if deps:
        print(f"  Received state from: {deps}")
    else:
        print(f"  Root node (receives only the task)")
    print()

=== State Sharing Analysis ===

Node: planner
  Declared share_output: ['research_questions', 'search_strategy']
  -> research_questions = <not present>
  -> search_strategy = <not present>
  Root node (receives only the task)

Node: researcher
  Declared share_output: ['findings', 'sources']
  -> findings = <not present>
  -> sources = <not present>
  Received state from: ['planner']

Node: writer
  Declared share_output: ['report']
  -> report = <not present>
  Received state from: ['researcher']



## 8. Error Handling

AWP DAG workflows support three error handling modes:

| Mode | Behavior |
|------|----------|
| `continue` | Log the error and proceed to the next agent (default) |
| `skip` | Skip downstream agents that depend on the failed agent |
| `abort` | Stop the entire workflow immediately |

The workflow also has timeout configuration:
- `per_agent` — maximum time each agent can run
- `total` — maximum time for the entire workflow
- `max_retries` — number of retry attempts per agent
- `retry_delay` — seconds between retries

In [14]:
# Inspect the error handling and timeout config for both workflows
print("=== Research Pipeline Error Config ===")
err = manifest.orchestration.execution.error_handling
timeout = manifest.orchestration.execution.timeout
print(f"  Error handling mode: {err.default}")
print(f"  Max retries: {err.max_retries}")
print(f"  Retry delay: {err.retry_delay}s")
print(f"  Timeout per agent: {timeout.per_agent}s")
print(f"  Timeout total: {timeout.total}s")
print()

print("=== Hello World Error Config ===")
hello_err = hello_manifest.orchestration.execution.error_handling
hello_timeout = hello_manifest.orchestration.execution.timeout
print(f"  Error handling mode: {hello_err.default}")
print(f"  Max retries: {hello_err.max_retries}")
print(f"  Retry delay: {hello_err.retry_delay}s")
print(f"  Timeout per agent: {hello_timeout.per_agent}s")
print(f"  Timeout total: {hello_timeout.total}s")
print()

# Show what happens if a result includes an error
print("=== Checking for errors in results ===")
for agent_name, agent_output in result.items():
    if isinstance(agent_output, dict):
        if "error" in agent_output:
            print(f"  {agent_name}: ERROR - {agent_output['error']}")
        else:
            conf = agent_output.get("confidence", "N/A")
            print(f"  {agent_name}: OK (confidence={conf})")

=== Research Pipeline Error Config ===
  Error handling mode: continue
  Max retries: 1
  Retry delay: 2.0s
  Timeout per agent: 60s
  Timeout total: 300s

=== Hello World Error Config ===
  Error handling mode: continue
  Max retries: 1
  Retry delay: 2.0s
  Timeout per agent: 30s
  Timeout total: 300s

=== Checking for errors in results ===
  planner: ERROR - Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401
  researcher: ERROR - Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401
  writer: ERROR - Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


## 9. Summary

### DAG Engine vs Delegation Loop Engine

| Feature | DAG Engine (A0-A1) | Delegation Loop (A2-A4) |
|---------|-------------------|------------------------|
| **Structure** | Fixed graph defined in YAML | Manager spawns workers dynamically |
| **Execution** | Topological sort, deterministic | Iterative loop, non-deterministic |
| **State** | Shared via `share_output` fields | Passed through delegation messages |
| **Parallelism** | Independent branches run in parallel | Workers can run concurrently |
| **Error handling** | continue / skip / abort | Budget-enforced termination |
| **Best for** | Pipelines, ETL, predictable tasks | Creative, exploratory, complex tasks |

### DAG Advantages

- **Predictable** — execution order is determined by the graph, not by LLM decisions
- **Debuggable** — you can inspect each agent's input/output at every step
- **Efficient** — no manager overhead; agents run directly in dependency order
- **Reproducible** — same graph + same input = same execution path
- **Simple to reason about** — visualize the flow as a directed graph

### When to Use DAG Workflows

Use DAG workflows when:
- The task has a known, fixed structure (e.g., plan -> research -> write)
- You need predictable execution and easy debugging
- Agent dependencies are clear at design time
- You want minimal LLM overhead (no manager agent)

Use delegation loop workflows when:
- The task requires dynamic decision-making
- The number of agents or steps is not known in advance
- You need recursive delegation or self-tooling capabilities